In [2]:
import pandas as pd
import numpy as np
import joblib
import os

# Load trained model
model = joblib.load("../models/final_xgboost_model.pkl")

# Load decision threshold
threshold = joblib.load("../models/decision_threshold.pkl")

print("Model loaded successfully!")
print("Decision Threshold:", threshold)

Model loaded successfully!
Decision Threshold: 0.4


In [3]:
sensor_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

print("Sensor features:")
for feature in sensor_features:
    print("-", feature)

Sensor features:
- Air temperature [K]
- Process temperature [K]
- Rotational speed [rpm]
- Torque [Nm]
- Tool wear [min]


In [4]:
print("Expected model features:")
print(model.get_booster().feature_names)

Expected model features:
['numerical__Air_temperature_K', 'numerical__Process_temperature_K', 'numerical__Rotational_speed_rpm', 'numerical__Torque_Nm', 'numerical__Tool_wear_min', 'numerical__Temperature_Difference_K', 'numerical__Mechanical_Power_kW', 'categorical__Type_H', 'categorical__Type_L', 'categorical__Type_M']


In [5]:
def prepare_machine_features(
    air_temperature,
    process_temperature,
    rotational_speed,
    torque,
    tool_wear,
    machine_type
):

    # Calculate engineered features
    temperature_difference = (
        process_temperature - air_temperature
    )

    mechanical_power = (
        torque * rotational_speed * (2 * np.pi / 60)
    ) / 1000

    # Prepare model input in exact training order
    input_data = pd.DataFrame([{
        "numerical__Air_temperature_K": air_temperature,
        "numerical__Process_temperature_K": process_temperature,
        "numerical__Rotational_speed_rpm": rotational_speed,
        "numerical__Torque_Nm": torque,
        "numerical__Tool_wear_min": tool_wear,
        "numerical__Temperature_Difference_K": temperature_difference,
        "numerical__Mechanical_Power_kW": mechanical_power,

        "categorical__Type_H": int(machine_type == "H"),
        "categorical__Type_L": int(machine_type == "L"),
        "categorical__Type_M": int(machine_type == "M")
    }])

    return input_data

In [8]:
# Select one original record
test_sample = X_test_xgb.iloc[0]

# Generate model input
sample_input = prepare_machine_features(
    air_temperature=test_sample["numerical__Air_temperature_K"],
    process_temperature=test_sample["numerical__Process_temperature_K"],
    rotational_speed=test_sample["numerical__Rotational_speed_rpm"],
    torque=test_sample["numerical__Torque_Nm"],
    tool_wear=test_sample["numerical__Tool_wear_min"],
    machine_type=(
        "H" if test_sample["categorical__Type_H"] == 1
        else "L" if test_sample["categorical__Type_L"] == 1
        else "M"
    )
)

print("Prepared machine input:")
display(sample_input)

print("Input shape:", sample_input.shape)
print("Feature order matches:", 
      list(sample_input.columns) == model.get_booster().feature_names)

Prepared machine input:


,numerical__Air_temperature_K,numerical__Process_temperature_K,numerical__Rotational_speed_rpm,numerical__Torque_Nm,numerical__Tool_wear_min,numerical__Temperature_Difference_K,numerical__Mechanical_Power_kW,categorical__Type_H,categorical__Type_L,categorical__Type_M
0,0.247681,-0.139294,-1.073963,2.265378,0.71241,-0.386975,-0.000255,0,1,0


Input shape: (1, 10)
Feature order matches: True


In [7]:
import pandas as pd

X_test_xgb = pd.read_csv(
    "../data/processed/X_test_final.csv"
)

# Clean feature names exactly as before
X_test_xgb.columns = (
    X_test_xgb.columns
    .str.replace(r"[\[\]<>]", "", regex=True)
    .str.replace(" ", "_")
)

print("Test dataset loaded successfully!")
print("Shape:", X_test_xgb.shape)
print(X_test_xgb.columns.tolist())

Test dataset loaded successfully!
Shape: (2000, 10)
['numerical__Air_temperature_K', 'numerical__Process_temperature_K', 'numerical__Rotational_speed_rpm', 'numerical__Torque_Nm', 'numerical__Tool_wear_min', 'numerical__Temperature_Difference_K', 'numerical__Mechanical_Power_kW', 'categorical__Type_H', 'categorical__Type_L', 'categorical__Type_M']


In [9]:
sample_probability = model.predict_proba(sample_input)[0][1]

sample_prediction = int(sample_probability >= threshold)

print("Failure Probability:",
      round(sample_probability * 100, 2), "%")

print("Machine Status:",
      "FAILURE RISK" if sample_prediction == 1 else "NORMAL")

Failure Probability: 0.97 %
Machine Status: NORMAL


In [10]:
import os

print("Available model files:")

for file in os.listdir("../models"):
    print("-", file)

Available model files:
- decision_threshold.pkl
- final_xgboost_model.pkl
- preprocessor.pkl
- preprocessor_fe.pkl
- shap_explainer.pkl


In [11]:
# Load feature-engineered preprocessor
preprocessor = joblib.load(
    "../models/preprocessor_fe.pkl"
)

print("Preprocessor loaded successfully!")

print("\nNumerical features:")
print(preprocessor.transformers_[0][2])

print("\nCategorical features:")
print(preprocessor.transformers_[1][2])

Preprocessor loaded successfully!

Numerical features:
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Temperature Difference [K]', 'Mechanical Power [kW]']

Categorical features:
['Type']


In [12]:
# Load raw engineered test dataset
X_test_engineered = pd.read_csv(
    "../data/processed/X_test_engineered.csv"
)

# Select one raw machine record
sample_raw = X_test_engineered.iloc[[0]]

print("Raw engineered input:")
display(sample_raw)

# Apply saved preprocessing
sample_processed = preprocessor.transform(sample_raw)

print("Processed input shape:", sample_processed.shape)

Raw engineered input:


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Temperature Difference [K],Mechanical Power [kW],Type
0,300.5,309.8,1345,62.7,153,9.3,8.831174,L


Processed input shape: (1, 10)


In [13]:
# Convert to DataFrame with exact model feature names
model_features = model.get_booster().feature_names

sample_processed_df = pd.DataFrame(
    sample_processed,
    columns=model_features
)

# Generate prediction
probability = model.predict_proba(
    sample_processed_df
)[0][1]

prediction = int(probability >= threshold)

print("Failure Probability:", round(probability * 100, 2), "%")

print("Machine Status:",
      "FAILURE RISK" if prediction == 1 else "NORMAL")

Failure Probability: 5.03 %
Machine Status: NORMAL


In [14]:
def predict_machine_failure(
    air_temperature,
    process_temperature,
    rotational_speed,
    torque,
    tool_wear,
    machine_type
):

    # 1. Validate inputs
    if machine_type not in ["L", "M", "H"]:
        raise ValueError("Machine type must be L, M or H.")

    if air_temperature <= 0 or process_temperature <= 0:
        raise ValueError("Temperatures must be positive.")

    if rotational_speed <= 0:
        raise ValueError("Rotational speed must be positive.")

    if torque < 0 or tool_wear < 0:
        raise ValueError("Torque and tool wear cannot be negative.")

    # 2. Feature engineering
    temperature_difference = (
        process_temperature - air_temperature
    )

    mechanical_power = (
        torque * rotational_speed * (2 * np.pi / 60)
    ) / 1000

    # 3. Create raw feature DataFrame
    machine_data = pd.DataFrame([{
        "Air temperature [K]": air_temperature,
        "Process temperature [K]": process_temperature,
        "Rotational speed [rpm]": rotational_speed,
        "Torque [Nm]": torque,
        "Tool wear [min]": tool_wear,
        "Temperature Difference [K]": temperature_difference,
        "Mechanical Power [kW]": mechanical_power,
        "Type": machine_type
    }])

    # 4. Apply saved preprocessing
    processed_data = preprocessor.transform(machine_data)

    # 5. Match XGBoost feature names
    processed_data = pd.DataFrame(
        processed_data,
        columns=model.get_booster().feature_names
    )

    # 6. Predict failure probability
    failure_probability = model.predict_proba(
        processed_data
    )[0][1]

    # 7. Apply decision threshold
    prediction = int(failure_probability >= threshold)

    # 8. Return results
    return {
        "failure_probability": round(
            float(failure_probability) * 100, 2
        ),
        "prediction": prediction,
        "machine_status": (
            "FAILURE RISK" if prediction == 1 else "NORMAL"
        ),
        "temperature_difference": round(
            temperature_difference, 2
        ),
        "mechanical_power": round(
            mechanical_power, 2
        ),
        "machine_type": machine_type
    }

print("Prediction function created successfully!")

Prediction function created successfully!


In [15]:
result = predict_machine_failure(
    air_temperature=300.1,
    process_temperature=310.2,
    rotational_speed=1500,
    torque=40,
    tool_wear=120,
    machine_type="L"
)

print("MACHINE PREDICTION REPORT")
print("------------------------")

for key, value in result.items():
    print(f"{key}: {value}")

MACHINE PREDICTION REPORT
------------------------
failure_probability: 0.07
prediction: 0
machine_status: NORMAL
temperature_difference: 10.1
mechanical_power: 6.28
machine_type: L


In [16]:
test_cases = [
    {
        "name": "Machine A",
        "air_temperature": 300.1,
        "process_temperature": 310.2,
        "rotational_speed": 1500,
        "torque": 40,
        "tool_wear": 50,
        "machine_type": "L"
    },
    {
        "name": "Machine B",
        "air_temperature": 302.0,
        "process_temperature": 311.0,
        "rotational_speed": 1400,
        "torque": 55,
        "tool_wear": 220,
        "machine_type": "M"
    }
]

results = []

for machine in test_cases:

    prediction = predict_machine_failure(
        air_temperature=machine["air_temperature"],
        process_temperature=machine["process_temperature"],
        rotational_speed=machine["rotational_speed"],
        torque=machine["torque"],
        tool_wear=machine["tool_wear"],
        machine_type=machine["machine_type"]
    )

    results.append({
        "Machine": machine["name"],
        "Failure Probability (%)": prediction["failure_probability"],
        "Status": prediction["machine_status"]
    })

display(pd.DataFrame(results))

,Machine,Failure Probability (%),Status
0,Machine A,0.05,NORMAL
1,Machine B,85.46,FAILURE RISK


In [17]:
def classify_risk(probability):

    if probability < 20:
        return "LOW"

    elif probability < 40:
        return "MEDIUM"

    else:
        return "HIGH"

In [18]:
def generate_maintenance_recommendation(
    result,
    tool_wear,
    torque,
    rotational_speed,
    temperature_difference
):

    recommendations = []

    risk = classify_risk(
        result["failure_probability"]
    )

    # Risk-based recommendations
    if risk == "HIGH":
        recommendations.append(
            "High failure risk: Arrange prompt machine inspection."
        )

    elif risk == "MEDIUM":
        recommendations.append(
            "Moderate failure risk: Schedule a maintenance inspection."
        )

    else:
        recommendations.append(
            "Low failure risk: Continue routine monitoring."
        )

    # Sensor-based recommendations
    if tool_wear >= 200:
        recommendations.append(
            "High tool wear: Inspect the cutting tool and assess replacement."
        )

    if torque >= 55:
        recommendations.append(
            "Elevated torque: Inspect mechanical load and drive components."
        )

    if rotational_speed <= 1300:
        recommendations.append(
            "Low rotational speed: Inspect motor and operating conditions."
        )

    if temperature_difference <= 8:
        recommendations.append(
            "Low temperature difference: Verify operating temperature sensors."
        )

    return {
        "risk_level": risk,
        "recommendations": recommendations
    }

In [19]:
for machine in test_cases:

    result = predict_machine_failure(
        air_temperature=machine["air_temperature"],
        process_temperature=machine["process_temperature"],
        rotational_speed=machine["rotational_speed"],
        torque=machine["torque"],
        tool_wear=machine["tool_wear"],
        machine_type=machine["machine_type"]
    )

    maintenance = generate_maintenance_recommendation(
        result=result,
        tool_wear=machine["tool_wear"],
        torque=machine["torque"],
        rotational_speed=machine["rotational_speed"],
        temperature_difference=(
            machine["process_temperature"]
            - machine["air_temperature"]
        )
    )

    print("\n", "=" * 50)
    print("MACHINE:", machine["name"])
    print("Failure Probability:", result["failure_probability"], "%")
    print("Risk Level:", maintenance["risk_level"])
    print("Maintenance Recommendations:")

    for recommendation in maintenance["recommendations"]:
        print("-", recommendation)


MACHINE: Machine A
Failure Probability: 0.05 %
Risk Level: LOW
Maintenance Recommendations:
- Low failure risk: Continue routine monitoring.

MACHINE: Machine B
Failure Probability: 85.46 %
Risk Level: HIGH
Maintenance Recommendations:
- High failure risk: Arrange prompt machine inspection.
- High tool wear: Inspect the cutting tool and assess replacement.
- Elevated torque: Inspect mechanical load and drive components.


In [20]:
def generate_machine_report(machine):

    result = predict_machine_failure(
        air_temperature=machine["air_temperature"],
        process_temperature=machine["process_temperature"],
        rotational_speed=machine["rotational_speed"],
        torque=machine["torque"],
        tool_wear=machine["tool_wear"],
        machine_type=machine["machine_type"]
    )

    maintenance = generate_maintenance_recommendation(
        result=result,
        tool_wear=machine["tool_wear"],
        torque=machine["torque"],
        rotational_speed=machine["rotational_speed"],
        temperature_difference=(
            machine["process_temperature"]
            - machine["air_temperature"]
        )
    )

    report = {
        "Machine ID": machine["name"],
        "Machine Type": machine["machine_type"],
        "Failure Probability (%)": result["failure_probability"],
        "Machine Status": result["machine_status"],
        "Risk Level": maintenance["risk_level"],
        "Temperature Difference (K)": result["temperature_difference"],
        "Mechanical Power (kW)": result["mechanical_power"],
        "Maintenance Recommendations": " | ".join(
            maintenance["recommendations"]
        )
    }

    return report

In [21]:
machine_reports = []

for machine in test_cases:
    report = generate_machine_report(machine)
    machine_reports.append(report)

report_df = pd.DataFrame(machine_reports)

display(report_df.T)

,0,1
Machine ID,Machine A,Machine B
Machine Type,L,M
Failure Probability (%),0.05,85.46
Machine Status,NORMAL,FAILURE RISK
Risk Level,LOW,HIGH
Temperature Difference (K),10.1,9.0
Mechanical Power (kW),6.28,8.06
Maintenance Recommendations,Low failure risk: Continue routine monitoring.,High failure risk: Arrange prompt machine insp...


In [22]:
os.makedirs("../reports", exist_ok=True)

report_df.to_csv(
    "../reports/machine_health_reports.csv",
    index=False
)

print("Machine health reports saved successfully!")

Machine health reports saved successfully!
